# AgriVision v8 — Skenario dataset: **lab saja** vs **gabungan** (PlantVillage + PlantDoc + Tomato-Village)

Notebook ini mengikuti arahan dosen: dataset **diatur dulu**, baru model dilatih.

**Apa yang dilakukan:**
1. Mengunduh tiga dataset: **PlantVillage** (foto lab), **PlantDoc** (foto internet/non-lab), **Tomato-Village** (foto kebun).
2. Memetakan semuanya ke **10 kelas** yang sama (9 penyakit + sehat). PlantDoc hanya punya 8 kelas, Tomato-Village 3 kelas; kelas yang tidak ada **tidak ditambah**, jumlahnya tetap dari PlantVillage saja.
3. Membuang foto yang **hampir kembar** (dHash) di dalam dan antar-dataset, supaya tidak ada foto latih yang bocor ke data uji.
4. Membagi **setiap kelas di setiap dataset** menjadi **70% latih : 10% validasi : 20% uji**. PlantVillage dibagi **per daun** (foto dari daun yang sama selalu berada di bagian yang sama).
5. Mencetak tabel jumlah data per kelas per dataset, dan per skenario.
6. Melatih MobileNetV2 dengan dua skenario, masing-masing **3 seed** (42, 1, 2):
   - **Skenario 1 — lab saja:** belajar dari bagian latih PlantVillage saja.
   - **Skenario 2 — gabungan:** belajar dari bagian latih PlantVillage + PlantDoc + Tomato-Village yang **digabung per kelas** (gabung polos, tanpa pengaturan porsi).
7. Menguji **kedua skenario pada data uji yang sama**, dilaporkan **per dataset** (uji lab, uji PlantDoc, uji Tomato-Village) dan **gabungan semua data uji**: akurasi, presisi, recall, F1 (makro dan per kelas), uji McNemar, confusion matrix.

Resep training sama dengan eksperimen di naskah (Tabel 2): MobileNetV2 224 px, tahap 1 kepala Adam 1e-3 ≤ 8 epoch, tahap 2 buka 60 lapisan atas (BatchNorm beku) Adam 1e-4 ≤ 15 epoch, augmentasi ringan, class weight, early stopping pada akurasi validasi skenario itu.

**Menjalankan:** Runtime → Change runtime type → **T4 GPU**, lalu Runtime → **Run all**. Perkiraan ±1,5–2,5 jam.
- Hasil tiap model langsung disimpan ke **Google Drive** (`MyDrive/AgriVision_v8`). Kalau sesi Colab putus, cukup **Run all lagi**: model yang sudah selesai otomatis dilewati.
- **Kalau hanya ingin melihat tabel dataset** (tanpa training): klik sel Step 8, lalu Runtime → **Run before**.
- Sel terakhir mengunduh `hasil_v8.zip` (tabel CSV, gambar confusion matrix, hasil mentah JSON).

In [ ]:
# Cek GPU aktif — kalau kosong/error: Runtime > Change runtime type > T4 GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Step 0 — Install, import, dan pengaturan

In [ ]:
!pip install -q datasets scikit-learn seaborn
import os, re, json, time, shutil, random, hashlib, subprocess, zipfile
from collections import Counter
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image

try:
    display
except NameError:          # di luar Jupyter
    display = print

# ---- pengaturan utama ----
SEEDS = [42, 1, 2]                                   # 3 kali latih per skenario, lalu dirata-rata
RATIO = {"train": 0.70, "val": 0.10, "test": 0.20}   # pembagian per kelas per dataset
SPLIT_SEED = 42                                      # pembagian data SAMA untuk semua seed training
IMG_SIZE = (224, 224)
BATCH = 32
EPOCHS_HEAD, EPOCHS_FT = 8, 15
NEAR_DUP_THR = 3                                     # jarak Hamming dHash <= 3 dianggap foto hampir kembar

POOL_DIR = "/content/pool_v8"     # semua foto (sudah 224 px) sebelum dibagi
DATA_DIR = "/content/data_v8"     # <sumber>/<train|val|test>/<kelas>/*.png
OUT_DIR = "/content/out_v8"       # tabel dan gambar hasil
USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/AgriVision_v8"

SAVE_DIR = "/content/simpan_v8"   # tanpa Drive: hasil per model & model Keras (tidak ikut zip)
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        SAVE_DIR = DRIVE_DIR
    except Exception as e:
        print("Google Drive tidak dipakai (hasil hanya di /content):", e)
for d in (SAVE_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)

random.seed(SPLIT_SEED); np.random.seed(SPLIT_SEED); tf.random.set_seed(SPLIT_SEED)
print("TensorFlow:", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))
print("Hasil disimpan ke:", SAVE_DIR)

## Step 1 — 10 kelas dan fungsi bantu

Urutan kelas sama dengan model di aplikasi (`label_map.json`, indeks 0–9).

In [ ]:
TOMATO_CLASSES = [
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy",
]
NICE = dict(zip(TOMATO_CLASSES, ["Bacterial spot", "Early blight", "Late blight", "Leaf mold", "Septoria",
                                 "Spider mites", "Target spot", "TYLCV", "ToMV", "Healthy"]))
CLASS_INDEX = {c: i for i, c in enumerate(TOMATO_CLASSES)}
SOURCES = {"pv": "PlantVillage (lab)", "pd": "PlantDoc", "tv": "Tomato-Village"}
SPLITS = ["train", "val", "test"]
SPLIT_ID = {"train": "latih", "val": "validasi", "test": "uji"}

def norm(s):
    return re.sub(r"[^a-z0-9]", "", s.lower())

CANON_BY_NORM = {norm(c): c for c in TOMATO_CLASSES}

def dhash(img, size=8):
    g = img.convert("L").resize((size + 1, size), Image.LANCZOS)
    a = np.asarray(g, dtype=np.int16)
    bits = (a[:, 1:] > a[:, :-1]).flatten()
    return int("".join("1" if b else "0" for b in bits), 2)

def popcount64(a):
    a = np.ascontiguousarray(np.atleast_1d(a).astype(np.uint64))
    return np.unpackbits(a.view(np.uint8).reshape(-1, 8), axis=1).sum(axis=1)

def ribuan(n):
    return f"{int(n):,}".replace(",", ".")

# POOL = semua foto dari ketiga dataset, sebelum dibagi.
# Tiap baris: id, src (pv/pd/tv), cls, group (foto satu grup selalu masuk bagian yang sama), h (dHash), path
POOL = []
shutil.rmtree(POOL_DIR, ignore_errors=True)
os.makedirs(POOL_DIR, exist_ok=True)

def add_to_pool(img, src, cls, group=None):
    i = len(POOL)
    img = img.convert("RGB")
    path = f"{POOL_DIR}/{src}_{i:06d}.png"
    img.resize(IMG_SIZE, Image.BILINEAR).save(path, compress_level=1)   # resize seperti di API
    POOL.append({"id": i, "src": src, "cls": cls, "group": group or f"{src}|{i}", "h": dhash(img), "path": path})

def pool_counts(title):
    df = pd.DataFrame([(r["src"], r["cls"]) for r in POOL], columns=["src", "cls"])
    t = df.value_counts().unstack(0).reindex(index=TOMATO_CLASSES, columns=list(SOURCES)).fillna(0).astype(int)
    t.index = [NICE[c] for c in t.index]
    t.columns = [SOURCES[s] for s in t.columns]
    t.loc["TOTAL"] = t.sum()
    print(title); display(t)

## Step 2 — PlantVillage (foto lab, 10 kelas)

Semua foto tomat berwarna diambil (split resmi PlantVillage **tidak** dipakai; nanti dibagi ulang 70/10/20 per kelas). Setiap foto diberi identitas **daun** (`leaf_id`) supaya foto dari daun yang sama tidak terpisah ke latih dan uji.

In [ ]:
# mohanty/PlantVillage dibaca sebagai berkas mentah (load_dataset() gagal untuk dataset berbasis skrip ini).
from huggingface_hub import hf_hub_download

REPO = "mohanty/PlantVillage"
zip_path = hf_hub_download(REPO, "data.zip", repo_type="dataset")   # +-2,2 GB
leaf_map = json.load(open(hf_hub_download(REPO, "leaf_grouping/leaf-map.json", repo_type="dataset"), encoding="utf-8"))
zf = zipfile.ZipFile(zip_path)
member_by_rel = {}
for n in zf.namelist():
    if "raw/color/Tomato" in n and not n.endswith("/"):
        member_by_rel[n[n.index("raw/color/"):]] = n
assert member_by_rel, "tidak ada raw/color/Tomato* di data.zip — cek zf.namelist()[:10]"

def leaf_id_for(filename, class_name):
    # meniru turunan leaf_id di plant_village.py
    name = filename.replace("_final_masked", "")
    if "___" in name:
        name = name.split("___")[-1]
    low = name.lower()
    if "copy" in low:
        name = name[: low.index("copy")]
    for ext in (".jpg", ".png"):
        if name.lower().endswith(ext):
            name = name[: -len(ext)]
    ident = name.strip().lower()
    sugg = leaf_map.get(ident)
    if isinstance(sugg, str):
        sugg = [sugg]
    if sugg:
        if len(sugg) == 1:
            return str(sugg[0])
        for sg in sugg:
            if class_name in str(sg):
                return str(sg)
    return f"fallback_{ident}"

t0 = time.time()
n_fallback = 0
for k, rel in enumerate(sorted(member_by_rel)):
    folder, fname = rel.split("/")[2], rel.split("/")[-1]
    if norm(folder) not in CANON_BY_NORM:
        continue
    cls = CANON_BY_NORM[norm(folder)]
    leaf = leaf_id_for(fname, folder)
    n_fallback += leaf.startswith("fallback_")
    with zf.open(member_by_rel[rel]) as fh:
        img = Image.open(fh).convert("RGB")
    add_to_pool(img, "pv", cls, f"pv|{cls}|{leaf}")
    if (k + 1) % 3000 == 0:
        print(f"  {k + 1} foto diproses ({time.time() - t0:.0f} dtk)")
n_pv = sum(r["src"] == "pv" for r in POOL)
print(f"PlantVillage tomat: {n_pv} foto; leaf_id dari nama berkas (fallback): {n_fallback}")

## Step 3 — PlantDoc (foto non-lab, 8 kelas)

Pemetaan nama kelas PlantDoc → 10 kelas kita. **Target spot** tidak ada di PlantDoc, **spider mites** hanya 2 foto (terlalu sedikit untuk dibagi) → kedua kelas itu tidak mendapat tambahan.

In [ ]:
from datasets import load_dataset

pdoc = load_dataset("Project-AgML/plant_doc_classification", split="train")
pdoc_names = pdoc.features["label"].names
PLANTDOC_TO_CANON = {
    "Tomato Early blight leaf": "Tomato___Early_blight",
    "Tomato Septoria leaf spot": "Tomato___Septoria_leaf_spot",
    "Tomato leaf": "Tomato___healthy",
    "Tomato leaf bacterial spot": "Tomato___Bacterial_spot",
    "Tomato leaf late blight": "Tomato___Late_blight",
    "Tomato leaf mosaic virus": "Tomato___Tomato_mosaic_virus",
    "Tomato leaf yellow virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato mold leaf": "Tomato___Leaf_Mold",
    # "Tomato two spotted spider mites leaf": hanya 2 gambar -> tidak dipakai
    # Target Spot: tidak ada di PlantDoc
}
missing = [k for k in PLANTDOC_TO_CANON if k not in pdoc_names]
assert not missing, f"nama kelas PlantDoc tidak ditemukan: {missing}. Daftar: {pdoc_names}"
pdoc_idx_to_canon = {pdoc_names.index(k): v for k, v in PLANTDOC_TO_CANON.items()}

for ex in pdoc.filter(lambda lab: lab in pdoc_idx_to_canon, input_columns=["label"]):
    add_to_pool(ex["image"], "pd", pdoc_idx_to_canon[ex["label"]])
print("PlantDoc tomat (dipetakan):", sum(r["src"] == "pd" for r in POOL))

## Step 4 — Tomato-Village (foto kebun, 3 kelas)

Hanya 3 kelas yang ada di 10 kelas kita: Early blight, Late blight, Healthy. Kelas lain di Tomato-Village (Leaf Miner, Spotted Wilt Virus, kekurangan Mg/N/K) **tidak ada** di PlantVillage/PlantDoc sehingga tidak dipakai. Split resmi Tomato-Village digabung lalu dibagi ulang 70/10/20.

In [ ]:
TV_FOLDER_TO_CLASS = {"Early_blight": "Tomato___Early_blight", "Late_blight": "Tomato___Late_blight", "Healthy": "Tomato___healthy"}
TV_BASE = "/content/ext/tv/Variant-a(Multiclass Classification)"
sparse = " ".join(f'"Variant-a(Multiclass Classification)/{sp}/{f}"' for sp in ["train", "val", "test"] for f in TV_FOLDER_TO_CLASS)
cmds = f"""
set -e
rm -rf /content/ext && mkdir -p /content/ext && cd /content/ext
git clone -q --filter=blob:none --no-checkout --depth 1 https://github.com/mamta-joshi-gehlot/Tomato-Village.git tv
cd tv
git sparse-checkout init --cone
git sparse-checkout set {sparse}
git checkout -q
"""
subprocess.run(["bash", "-c", cmds], check=True)

for sp in ["train", "val", "test"]:
    for folder, cls in TV_FOLDER_TO_CLASS.items():
        d = f"{TV_BASE}/{sp}/{folder}"
        for f in sorted(os.listdir(d)):
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                add_to_pool(Image.open(f"{d}/{f}"), "tv", cls)
print("Tomato-Village (3 kelas):", sum(r["src"] == "tv" for r in POOL))

## Step 5 — Buang foto yang hampir kembar (sebelum dibagi)

Supaya data uji benar-benar "belum pernah dilihat" model:
1. **Di dalam** PlantDoc dan Tomato-Village: foto yang hampir kembar dengan foto lain disisakan satu. Kalau dua foto kembar tapi labelnya berbeda, **keduanya** dibuang.
2. **Antar-dataset**: foto PlantDoc yang kembar dengan foto PlantVillage dibuang (berarti sebenarnya foto lab), lalu foto Tomato-Village yang kembar dengan PlantVillage/PlantDoc dibuang.

(PlantVillage tidak perlu: foto dari daun yang sama sudah dijaga lewat pembagian per daun.)

In [ ]:
pool_counts("SEBELUM dibersihkan:")

def near_dup_within(src):
    items = [r for r in POOL if r["src"] == src]
    kept_h = np.zeros(len(items), dtype=np.uint64)
    kept = []
    drop = set()
    for r in items:
        if kept:
            d = popcount64(kept_h[: len(kept)] ^ np.uint64(r["h"]))
            hit = np.where(d <= NEAR_DUP_THR)[0]
            if len(hit):
                drop.add(r["id"])
                for j in hit:
                    if kept[j]["cls"] != r["cls"]:      # kembar tapi beda label -> buang dua-duanya
                        drop.add(kept[j]["id"])
                continue
        kept_h[len(kept)] = np.uint64(r["h"])
        kept.append(r)
    return drop

def near_dup_against(src, ref_srcs, already):
    ref = np.array([r["h"] for r in POOL if r["src"] in ref_srcs and r["id"] not in already], dtype=np.uint64)
    return {r["id"] for r in POOL
            if r["src"] == src and r["id"] not in already and len(ref) and popcount64(ref ^ np.uint64(r["h"])).min() <= NEAR_DUP_THR}

DROP = set()
reasons = {}
for src in ["pd", "tv"]:
    d = near_dup_within(src); reasons[f"{SOURCES[src]}: kembar di dalam dataset"] = len(d); DROP |= d
d = near_dup_against("pd", {"pv"}, DROP); reasons["PlantDoc: kembar dengan PlantVillage"] = len(d); DROP |= d
d = near_dup_against("tv", {"pv", "pd"}, DROP); reasons["Tomato-Village: kembar dengan PlantVillage/PlantDoc"] = len(d); DROP |= d
for k, v in reasons.items():
    print(f"  dibuang — {k}: {v}")

for r in POOL:
    if r["id"] in DROP:
        os.remove(r["path"])
POOL = [r for r in POOL if r["id"] not in DROP]
pool_counts("SESUDAH dibersihkan:")

## Step 6 — Bagi 70% latih : 10% validasi : 20% uji, **per kelas per dataset**

Untuk setiap pasangan (dataset, kelas), grup foto diacak dengan seed tetap lalu dimasukkan ke uji sampai ±20%, ke validasi sampai ±10%, sisanya latih. Satu grup = satu daun (PlantVillage) atau satu foto (PlantDoc, Tomato-Village). Pembagian ini **sama untuk semua seed training**, jadi kedua skenario diuji dengan data uji yang persis sama.

In [ ]:
def split_one(records, rng):
    groups = {}
    for r in records:
        groups.setdefault(r["group"], []).append(r)
    keys = sorted(groups)
    rng.shuffle(keys)
    n = len(records)
    target_test = max(1, round(RATIO["test"] * n)) if n >= 2 else 0
    target_val = round(RATIO["val"] * n)
    out = {sp: [] for sp in SPLITS}
    for k in keys:
        g = groups[k]
        if len(out["test"]) < target_test:
            out["test"] += g
        elif len(out["val"]) < target_val:
            out["val"] += g
        else:
            out["train"] += g
    return out

shutil.rmtree(DATA_DIR, ignore_errors=True)
for src in SOURCES:
    for sp in SPLITS:
        for c in TOMATO_CLASSES:
            os.makedirs(f"{DATA_DIR}/{src}/{sp}/{c}", exist_ok=True)

for src in SOURCES:
    for c in TOMATO_CLASSES:
        recs = [r for r in POOL if r["src"] == src and r["cls"] == c]
        if not recs:
            continue
        parts = split_one(recs, random.Random(f"{SPLIT_SEED}|{src}|{c}"))
        for sp, rs in parts.items():
            for r in rs:
                r["split"] = sp
                dst = f"{DATA_DIR}/{src}/{sp}/{c}/{src}_{r['id']:06d}.png"
                shutil.move(r["path"], dst)
                r["path"] = dst

# Pengecekan: satu grup (daun) tidak boleh muncul di dua bagian
for src in SOURCES:
    seen = {}
    for r in POOL:
        if r["src"] == src:
            seen.setdefault(r["group"], set()).add(r["split"])
    bocor = [g for g, s in seen.items() if len(s) > 1]
    assert not bocor, f"{src}: {len(bocor)} grup muncul di lebih dari satu bagian"
print("OK — tidak ada daun/foto yang sama di dua bagian (latih/validasi/uji)")

# Sidik jari pembagian: memastikan sesi Colab berikutnya (lanjut training) memakai pembagian yang sama persis
manifest = pd.DataFrame([{"sumber": r["src"], "bagian": r["split"], "kelas": r["cls"], "grup": r["group"],
                          "dhash": f"{r['h']:016x}", "berkas": os.path.basename(r["path"])} for r in POOL])
manifest = manifest.sort_values(["sumber", "bagian", "kelas", "dhash", "berkas"]).reset_index(drop=True)
FINGERPRINT = hashlib.sha1(manifest[["sumber", "bagian", "kelas", "dhash"]].to_csv(index=False).encode()).hexdigest()
manifest.to_csv(f"{OUT_DIR}/manifest_pembagian.csv", index=False)
fp_path = f"{SAVE_DIR}/split_fingerprint.txt"
if os.path.exists(fp_path):
    old = open(fp_path).read().strip()
    assert old == FINGERPRINT, ("Pembagian data BERBEDA dari sesi sebelumnya (dataset sumber berubah?). "
                                f"Hapus folder {SAVE_DIR} di Drive untuk mulai dari awal.")
    print("Pembagian sama dengan sesi sebelumnya:", FINGERPRINT[:12])
else:
    open(fp_path, "w").write(FINGERPRINT)
    print("Sidik jari pembagian:", FINGERPRINT[:12])

## Step 7 — Tabel dataset (untuk naskah)

- **Tabel A** — jumlah foto per kelas di setiap dataset, format `latih / validasi / uji`.
- **Tabel B** — jumlah foto per kelas di setiap **skenario**. Skenario 1 hanya PlantVillage; skenario 2 menjumlahkan ketiga dataset per kelas. **Bagian uji dipakai oleh kedua skenario**, dan nilainya dilaporkan per dataset.

In [ ]:
CNT = Counter((r["src"], r["split"], r["cls"]) for r in POOL)

def n(src, sp, c=None):
    if c is None:
        return sum(CNT[(src, sp, cc)] for cc in TOMATO_CLASSES)
    return CNT[(src, sp, c)]

def tri(a, b, c):
    return "–" if a + b + c == 0 else f"{ribuan(a)} / {ribuan(b)} / {ribuan(c)}"

# ---- Tabel A: per dataset ----
rows = []
for c in TOMATO_CLASSES + ["TOTAL"]:
    row = {"Kelas": "Total" if c == "TOTAL" else NICE[c]}
    for src, name in SOURCES.items():
        cc = None if c == "TOTAL" else c
        row[name] = tri(n(src, "train", cc), n(src, "val", cc), n(src, "test", cc))
    rows.append(row)
pct = {"Kelas": "Persentase"}
for src, name in SOURCES.items():
    tot = sum(n(src, sp) for sp in SPLITS)
    pct[name] = " / ".join(f"{n(src, sp) / tot * 100:.1f}%".replace(".", ",") for sp in SPLITS)
rows.append(pct)
TABEL_A = pd.DataFrame(rows).set_index("Kelas")
print("Tabel A. Jumlah foto per kelas per dataset (latih / validasi / uji)")
display(TABEL_A)

# ---- Tabel B: per skenario ----
SCENARIOS = {"lab_saja": ["pv"], "gabungan": ["pv", "pd", "tv"]}
SCEN_NAME = {"lab_saja": "Skenario 1: lab saja", "gabungan": "Skenario 2: gabungan"}
rows = []
for c in TOMATO_CLASSES + ["TOTAL"]:
    cc = None if c == "TOTAL" else c
    row = {"Kelas": "Total" if c == "TOTAL" else NICE[c]}
    for scen, srcs in SCENARIOS.items():
        row[SCEN_NAME[scen]] = tri(*[sum(n(s, sp, cc) for s in srcs) for sp in SPLITS])
    if c != "TOTAL":
        extra = [SOURCES[s] for s in ["pd", "tv"] if n(s, "train", c) > 0]
        row["Tambahan dari"] = " + ".join(extra) if extra else "tetap (tidak ada data lain)"
    else:
        row["Tambahan dari"] = ""
    rows.append(row)
TABEL_B = pd.DataFrame(rows).set_index("Kelas")
print("\nTabel B. Jumlah foto per kelas per skenario (latih / validasi / uji)")
display(TABEL_B)

# Versi angka (untuk diolah di Excel)
num = pd.DataFrame([{"kelas": NICE[c], "sumber": SOURCES[s], "bagian": SPLIT_ID[sp], "jumlah": n(s, sp, c)}
                    for c in TOMATO_CLASSES for s in SOURCES for sp in SPLITS])
num.to_csv(f"{OUT_DIR}/tabel_A_angka.csv", index=False)
TABEL_A.to_csv(f"{OUT_DIR}/tabel_A_per_dataset.csv")
TABEL_B.to_csv(f"{OUT_DIR}/tabel_B_per_skenario.csv")
print("\nTersimpan:", sorted(os.listdir(OUT_DIR)))

## Step 8 — Pipeline data, model MobileNetV2, dan prosedur latih

Resep sama dengan Tabel 2 naskah. Validasi (untuk early stopping) memakai bagian validasi **dari sumber yang dipakai skenario itu**: skenario 1 hanya validasi PlantVillage (model lab saja tidak boleh "melihat" data lapangan sama sekali), skenario 2 validasi ketiga dataset.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

AUTOTUNE = tf.data.AUTOTUNE
TEST_SETS = {"uji_lab": ["pv"], "uji_plantdoc": ["pd"], "uji_tomato_village": ["tv"], "uji_gabungan": ["pv", "pd", "tv"]}
TEST_NAME = {"uji_lab": "Uji lab (PlantVillage)", "uji_plantdoc": "Uji PlantDoc",
             "uji_tomato_village": "Uji Tomato-Village", "uji_gabungan": "Uji gabungan (semua)"}

def gather(srcs, sp):
    paths, labels = [], []
    for s in srcs:
        for c in TOMATO_CLASSES:
            d = f"{DATA_DIR}/{s}/{sp}/{c}"
            for f in sorted(os.listdir(d)):
                paths.append(f"{d}/{f}")
                labels.append(CLASS_INDEX[c])
    return paths, np.array(labels, dtype=np.int32)

def _load(path, label):
    img = tf.io.decode_png(tf.io.read_file(path), channels=3)
    return tf.ensure_shape(img, IMG_SIZE + (3,)), label

def _to_f32(x, y):
    return tf.cast(x, tf.float32), y

def make_ds(paths, labels, train):
    if train:   # acak urutan sekali (berkas tersusun per kelas), lalu diacak lagi tiap epoch
        order = np.random.RandomState(SPLIT_SEED).permutation(len(paths))
        paths, labels = [paths[i] for i in order], labels[order]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels)).map(_load, num_parallel_calls=AUTOTUNE).cache()
    if train:
        ds = ds.shuffle(4000, seed=SPLIT_SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH).map(_to_f32, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

_DS = {}
def get_ds(kind, name):
    # dataset dibuat sekali dan dipakai ulang untuk semua seed (cache di RAM tidak menumpuk)
    key = (kind, name)
    if key not in _DS:
        if kind == "train":
            p, y = gather(SCENARIOS[name], "train"); _DS[key] = (make_ds(p, y, True), y)
        elif kind == "val":
            p, y = gather(SCENARIOS[name], "val"); _DS[key] = (make_ds(p, y, False), y)
        else:
            p, y = gather(TEST_SETS[name], "test"); _DS[key] = (make_ds(p, y, False), y)
    return _DS[key]

def class_weight_for(y):
    counts = np.bincount(y, minlength=len(TOMATO_CLASSES)).astype(float)
    return {i: float(counts.sum() / (len(counts) * max(cnt, 1))) for i, cnt in enumerate(counts)}

def make_augment():
    return tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal_and_vertical"),
        tf.keras.layers.RandomRotation(0.1),
        tf.keras.layers.RandomZoom(0.15),
        tf.keras.layers.RandomTranslation(0.05, 0.05),
        tf.keras.layers.RandomBrightness(0.15, value_range=(0, 255)),
        tf.keras.layers.RandomContrast(0.15),
    ], name="augment")

def build_mobilenet():
    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)   # input mentah 0-255 -> [-1, 1]
    base = tf.keras.applications.MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
    base.trainable = False
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(len(TOMATO_CLASSES), activation="softmax")(x)
    return tf.keras.Model(inputs, outputs, name="mobilenetv2_tomato"), base

def compile_model(model, lr):
    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])

def metrics_of(y_true, y_pred):
    present = sorted(set(y_true.tolist()))   # kelas yang ada di data uji ini
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=present, zero_division=0)
    return {
        "n": int(len(y_true)), "accuracy": float((y_true == y_pred).mean()),
        "macro_precision": float(p.mean()), "macro_recall": float(r.mean()), "macro_f1": float(f.mean()),
        "per_class": {TOMATO_CLASSES[k]: {"precision": float(p[i]), "recall": float(r[i]), "f1": float(f[i]), "n": int(s[i])}
                      for i, k in enumerate(present)},
    }

def train_and_eval(scen, seed):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    train_ds, y_train = get_ds("train", scen)
    val_ds, _ = get_ds("val", scen)
    cw = class_weight_for(y_train)
    core, base = build_mobilenet()
    model = tf.keras.Sequential([tf.keras.Input(shape=IMG_SIZE + (3,)), make_augment(), core], name="train_wrapper")
    stop = lambda p: [tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=p, restore_best_weights=True)]
    t0 = time.time()
    compile_model(model, 1e-3)
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, class_weight=cw, callbacks=stop(3), verbose=0)
    base.trainable = True
    for layer in base.layers[:-60]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
    compile_model(model, 1e-4)
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT, class_weight=cw, callbacks=stop(4), verbose=0)
    info = {"epochs": [len(h1.history["loss"]), len(h2.history["loss"])],
            "best_val_acc": float(max(h1.history["val_accuracy"] + h2.history["val_accuracy"])),
            "n_train": int(len(y_train)), "minutes": round((time.time() - t0) / 60, 1)}
    res = {"scenario": scen, "seed": seed, "fingerprint": FINGERPRINT, "info": info, "metrics": {}, "pred": {}, "y_true": {}, "conf": {}}
    for t in TEST_SETS:
        ds, y = get_ds("test", t)
        probs = core.predict(ds, verbose=0)
        pred = probs.argmax(1)
        res["metrics"][t] = metrics_of(y, pred)
        res["pred"][t] = "".join(map(str, pred.tolist()))
        res["y_true"][t] = "".join(map(str, y.tolist()))
        res["conf"][t] = float(probs.max(1).mean())
    os.makedirs(f"{SAVE_DIR}/models", exist_ok=True)
    core.save(f"{SAVE_DIR}/models/{scen}_seed{seed}.keras")
    del model, core
    return res

for scen in SCENARIOS:
    print(f"{SCEN_NAME[scen]}: {len(get_ds('train', scen)[1])} foto latih, {len(get_ds('val', scen)[1])} validasi")
for t in TEST_SETS:
    print(f"{TEST_NAME[t]}: {len(get_ds('test', t)[1])} foto")

## Step 9 — Latih 2 skenario × 3 seed

Urutan: seed 42 (skenario 1, skenario 2), lalu seed 1, lalu seed 2 — jadi kalau sesi putus di tengah, hasil yang sudah ada tetap seimbang antar-skenario. Hasil tiap model disimpan ke Drive (`runs/`), model Keras ke `models/`. **Run all lagi** setelah sesi putus akan melanjutkan dari yang belum selesai.

In [ ]:
RUN_DIR = f"{SAVE_DIR}/runs"
os.makedirs(RUN_DIR, exist_ok=True)
RESULTS = {}

t_all = time.time()
for seed in SEEDS:
    for scen in SCENARIOS:
        path = f"{RUN_DIR}/{scen}_seed{seed}.json"
        if os.path.exists(path):
            res = json.load(open(path))
            assert res["fingerprint"] == FINGERPRINT, f"{path} dibuat dari pembagian data lain"
            RESULTS[(scen, seed)] = res
            print(f"[{SCEN_NAME[scen]} | seed {seed}] sudah ada di Drive — dilewati")
            continue
        res = train_and_eval(scen, seed)
        json.dump(res, open(path, "w"))
        RESULTS[(scen, seed)] = res
        m = res["metrics"]
        print(f"[{SCEN_NAME[scen]} | seed {seed}] epoch {res['info']['epochs']} | "
              + " | ".join(f"{TEST_NAME[t]} {m[t]['accuracy'] * 100:.2f}%" for t in TEST_SETS)
              + f" ({res['info']['minutes']} mnt)")
print(f"Selesai ({(time.time() - t_all) / 60:.0f} menit). Model selesai: {len(RESULTS)} dari {len(SEEDS) * len(SCENARIOS)}")

## Step 10 — Hasil: tabel perbandingan, per kelas, McNemar, confusion matrix

- **Tabel C** — akurasi rata-rata ± simpangan baku (3 seed) per data uji.
- **Tabel D** — F1 makro. Presisi/recall/F1 makro dihitung hanya atas kelas yang ada di data uji itu (PlantDoc 8 kelas, Tomato-Village 3 kelas), tetapi model tetap memilih dari 10 kelas — jawaban kelas lain dihitung salah.
- **Tabel E** — uji McNemar per seed: apakah beda skenario 1 vs 2 pada data uji yang sama signifikan (p < 0,05).
- **Tabel F** — presisi/recall/F1 per kelas (rata-rata 3 seed).
- Confusion matrix (seed pertama) disimpan sebagai gambar.

In [ ]:
from scipy.stats import binomtest
import matplotlib.pyplot as plt
import seaborn as sns

done_seeds = [s for s in SEEDS if all((sc, s) in RESULTS for sc in SCENARIOS)]
assert done_seeds, "belum ada seed yang selesai untuk kedua skenario"
print("Seed yang dipakai di tabel:", done_seeds)

def koma(x, d=2):
    return f"{x:.{d}f}".replace(".", ",")

def ms(vals):
    v = np.array(vals) * 100
    return koma(v.mean()) if len(v) == 1 else f"{koma(v.mean())} ± {koma(v.std())}"   # simpangan baku populasi

def table_metric(key):
    return pd.DataFrame({TEST_NAME[t]: {SCEN_NAME[sc]: ms([RESULTS[(sc, s)]["metrics"][t][key] for s in done_seeds])
                                        for sc in SCENARIOS} for t in TEST_SETS})

TABEL_C = table_metric("accuracy")
TABEL_D = table_metric("macro_f1")
print(f"Tabel C. Akurasi (%) — rata-rata ± simpangan baku, {len(done_seeds)} seed")
display(TABEL_C)
print(f"\nTabel D. F1 makro (%) — rata-rata ± simpangan baku, {len(done_seeds)} seed")
display(TABEL_D)

# ---- McNemar (exact) skenario 1 vs 2, per seed, per data uji ----
rows = []
for s in done_seeds:
    for t in TEST_SETS:
        y = np.array(list(RESULTS[("lab_saja", s)]["y_true"][t]), dtype=int)
        p1 = np.array(list(RESULTS[("lab_saja", s)]["pred"][t]), dtype=int)
        p2 = np.array(list(RESULTS[("gabungan", s)]["pred"][t]), dtype=int)
        b = int(((p1 == y) & (p2 != y)).sum())   # benar di skenario 1 saja
        c = int(((p1 != y) & (p2 == y)).sum())   # benar di skenario 2 saja
        p = binomtest(min(b, c), b + c, 0.5).pvalue if b + c > 0 else 1.0
        rows.append({"Seed": s, "Data uji": TEST_NAME[t], "Benar hanya di S1": b, "Benar hanya di S2": c,
                     "p": f"{p:.2e}" if p < 0.001 else koma(p, 3), "Signifikan (p<0,05)": "ya" if p < 0.05 else "tidak"})
TABEL_E = pd.DataFrame(rows)
print("\nTabel E. Uji McNemar skenario 1 vs skenario 2")
display(TABEL_E)

# ---- Per kelas ----
rows = []
for t in ["uji_lab", "uji_plantdoc", "uji_tomato_village"]:
    for c in TOMATO_CLASSES:
        row = None
        for sc in SCENARIOS:
            per = [RESULTS[(sc, s)]["metrics"][t]["per_class"].get(c) for s in done_seeds]
            if per[0] is None:
                continue
            row = row or {"Data uji": TEST_NAME[t], "Kelas": NICE[c], "n": per[0]["n"]}
            for k, lab in [("precision", "P"), ("recall", "R"), ("f1", "F1")]:
                row[f"{lab} {'S1' if sc == 'lab_saja' else 'S2'}"] = koma(np.mean([x[k] for x in per]) * 100, 1)
        if row:
            rows.append(row)
TABEL_F = pd.DataFrame(rows)
print("\nTabel F. Presisi (P), recall (R), F1 per kelas, rata-rata seed (%) — S1 = lab saja, S2 = gabungan")
display(TABEL_F)

for name, tab in [("C_akurasi", TABEL_C), ("D_f1_makro", TABEL_D), ("E_mcnemar", TABEL_E), ("F_per_kelas", TABEL_F)]:
    tab.to_csv(f"{OUT_DIR}/tabel_{name}.csv", index=name in ("C_akurasi", "D_f1_makro"))

# ---- Confusion matrix (seed pertama) ----
s0 = done_seeds[0]
fig, axes = plt.subplots(2, 3, figsize=(21, 12))
for i, sc in enumerate(SCENARIOS):
    for j, t in enumerate(["uji_lab", "uji_plantdoc", "uji_tomato_village"]):
        y = np.array(list(RESULTS[(sc, s0)]["y_true"][t]), dtype=int)
        p = np.array(list(RESULTS[(sc, s0)]["pred"][t]), dtype=int)
        cm = confusion_matrix(y, p, labels=list(range(len(TOMATO_CLASSES))))
        present = sorted(set(y.tolist()))
        sns.heatmap(cm[present], annot=True, fmt="d", cmap="Greens", cbar=False, ax=axes[i, j],
                    xticklabels=[NICE[c] for c in TOMATO_CLASSES], yticklabels=[NICE[TOMATO_CLASSES[k]] for k in present])
        axes[i, j].set_title(f"{SCEN_NAME[sc]} — {TEST_NAME[t]} (seed {s0})")
        axes[i, j].set_xlabel("Prediksi"); axes[i, j].set_ylabel("Label asli")
        axes[i, j].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/confusion_matrix_seed{s0}.png", dpi=130)
plt.show()

# ---- Ringkasan ----
summary = {"seeds": done_seeds, "fingerprint": FINGERPRINT, "ratio": RATIO,
           "akurasi": {sc: {t: [RESULTS[(sc, s)]["metrics"][t]["accuracy"] for s in done_seeds] for t in TEST_SETS} for sc in SCENARIOS},
           "f1_makro": {sc: {t: [RESULTS[(sc, s)]["metrics"][t]["macro_f1"] for s in done_seeds] for t in TEST_SETS} for sc in SCENARIOS}}
json.dump(summary, open(f"{OUT_DIR}/ringkasan_v8.json", "w"), indent=2)
json.dump({f"{sc}_seed{s}": RESULTS[(sc, s)] for sc in SCENARIOS for s in done_seeds}, open(f"{OUT_DIR}/hasil_mentah_v8.json", "w"))
print("\nRingkasan akurasi rata-rata:")
for t in TEST_SETS:
    a1 = np.mean(summary["akurasi"]["lab_saja"][t]) * 100
    a2 = np.mean(summary["akurasi"]["gabungan"][t]) * 100
    print(f"  {TEST_NAME[t]:24s}: lab saja {koma(a1)}% -> gabungan {koma(a2)}% (selisih {'+' if a2 >= a1 else ''}{koma(a2 - a1)} poin)")

## Step 11 — Simpan ke Drive dan unduh hasil

In [ ]:
for f in os.listdir(OUT_DIR):
    shutil.copy(f"{OUT_DIR}/{f}", f"{SAVE_DIR}/{f}")
shutil.make_archive("/content/hasil_v8", "zip", OUT_DIR)
print("Isi hasil_v8.zip:", sorted(os.listdir(OUT_DIR)))
try:
    from google.colab import files
    files.download("/content/hasil_v8.zip")
except Exception as e:
    print("Unduh manual dari panel Files (/content/hasil_v8.zip):", e)